In [31]:
from Bio import SeqIO
from Bio.Seq import Seq

def translate_sequence(dna_sequence):
    return dna_sequence.translate()

def parse_genbank_translate(genbank_file):
    protein_sequences = []
    for record in SeqIO.parse(genbank_file, "genbank"):
        for feature in record.features:
            if feature.type == "gene":
                gene_seq = feature.location.extract(record.seq)
                protein_seq = translate_sequence(gene_seq)
    return protein_seq

genbank_file = "../data/PacBio_amplicon.gb"
protein_seq = parse_genbank_translate(genbank_file)
print(protein_seq)


DKICLGHHAVSNGTKVNTLTERGVEVVNATETVERTNIPRICSKGKRTVDLGQCGLLGTITGPPQCDQFLEFSADLIIERREGSDVCYPGKFVNEEALRQILRESGGIDKEAMGFTYSGIRTNGATSACRRSGSSFYAEMKWLLSNTDNAAFPQMTKSYKNTRKSPALIVWGIHHSVSTAEQTKLYGSGNKLVTVGSSNYQQSFVPSPGARPQVNGLSGRIDFHWLMLNPNDTVTFSFNGAFIAPDRASFLRGKSMGIQSGVQVDANCEGDCYHSGGTIISNLPFQNIDSRAVGKCPRYVKQRSLLLATGMKNVPEIPKGRGLFGAIAGFIENGWEGLIDGWYGFRHQNAQGEGTAADYKSTQSAIDQITGKLNRLIEKTNQQFELIDNEFNEVEKQIGNVINWTRDSITEVWSYNAELLVAMENQHTIDLADSEMDKLYERVKRQLRENAEEDGTGCFEIFHKCDDDCMASIRNNTYDHSKYREEAMQNRIQIDPVKLSSGYKDV


/home/jahn2/miniconda3/envs/dms-vep-pipeline-3/lib/python3.11/site-packages/Bio/GenBank/Scanner.py:1528: BiopythonParserWarning: Attempting to parse malformed locus line:
'LOCUS       PacBio_amplicon        2407 bp DNA     linear   UNA 23-SEP-2024\n'
Found locus 'PacBio_amplicon' size '2407' residue_type 'DNA'
Some fields may be wrong.
  warnings.warn(


### Make `site_numbering_map.csv`

In [32]:
import pandas as pd

# These RBS sites compiled from:
# https://www.ncbi.nlm.nih.gov/pmc/articles/PMC11149724/

RBS_loop_130 = [125, 126, 127, 128]
RBS_loop_150 = [144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154]
RBS_loop_190 = [181, 182, 183, 184, 185, 186, 187, 188, 189]
RBS_loop_220 = [212, 213, 214, 215, 216, 217, 218, 219]
RBS_base = [88,142,174,186] #this was converted from figure1 of the paper

# These are epitopes based on:
# https://www.ncbi.nlm.nih.gov/pmc/articles/PMC11149724/

epitope_A = list(range(111,136))
epitope_B = list(range(144,148)) + list(range( 150,152)) + list(range(177,186)) + list(range(187,191))
epitope_C = list(range(39,41)) + list(range(43,45)) + list(range(261,269)) + [269]
epitope_D = [158] + list(range( 192,200)) + [205] + list(range(207,212)) + list(range(213,219)) + [233]
epitope_E = list(range( 52,54)) + [65] + list(range( 67,73 )) + list(range(80,83 ))


def create_site_map(protein_seq):
    sequential_site = list(range(1, len(protein_seq)+ 1))
    reference_site = list(range(9, len(protein_seq)+ 9))  # Start reference_site at 1 as this is H3 numbering
    sequential_wt = list(protein_seq)


    print(len(protein_seq))
    
    def assign_epitope_region(reference_site):
    # Epitope assignments are obtained from Welsh et al. 2023
        if reference_site in epitope_A:
            return 'epitope-A'
        elif reference_site in epitope_B:
            return 'epitope-B'
        elif reference_site in epitope_C:
            return 'epitope-C'
        elif reference_site in epitope_D:
            return 'epitope-D'
        elif reference_site in epitope_E:
            return 'epitope-E'
        elif 1 <= reference_site <= 317:
            return 'HA1'
        elif 318 <= reference_site <= 490:
            return 'HA2'
        else:
            return 'Other'

    def assign_rbs_region(reference_site):
        if reference_site in RBS_loop_130:
            return 'RBS 130-loop'
        elif reference_site in RBS_loop_150:
            return 'RBS 150-loop'
        elif reference_site in RBS_loop_190:
            return 'RBS 190-loop'
        elif reference_site in RBS_loop_220:
            return 'RBS 220-loop'
        elif reference_site in RBS_base:
            return 'RBS base'
        else:
            return 'outside RBS'
    print (len(sequential_site))
    print (len(reference_site))
    print (len(sequential_wt))

    df = pd.DataFrame({
        "sequential_site": sequential_site,
        "reference_site": reference_site,
        "sequential_wt": sequential_wt
    })
    
    print ('here')
    df['region'] = df['reference_site'].apply(assign_epitope_region)
    df['rbs_region'] = df['reference_site'].apply(assign_rbs_region)
    
    return df

df = create_site_map(protein_seq)
df.to_csv('../data/site_numbering_map.csv', index=False)

506
506
506
506
here


### Make `mutation_design_classification.csv`

In [33]:
import numpy as np
import pandas as pd


vp = pd.read_csv('twist_qc_reports/Final_QC_Report_Q-367972VariantProportion.csv')
vp['variant_proportion'] = vp['variant_proportion'].str.replace('%', '').astype(float)

AA=['A','C','D','E','F','G','H','I','K','L','M','N','P','Q','R','S','T','V','W','Y'];

missing_sites = vp.query(
    'variant_proportion == 0.0'
)[['AA Position', 'wt_codon', 'wt_aa']].drop_duplicates()

missing_mutations = vp.query(
    'variant_proportion <= 1.0'
).query(
    '`AA Position` not in @missing_sites["AA Position"].to_numpy()'
).assign(
    mutation = lambda x: [
        str(wt)+str(site)+str(mut) for wt, site, mut in zip(x['wt_aa'], x['AA Position'], x['variant_aa'])
    ]
).reset_index(drop=True)
#print(missing_mutations.head())
#print(list(missing_mutations['mutation']))

stop_positions=list(np.arange(19,59,2))
#print(stop_positions)
assert len(stop_positions) == 20

sequential_site=[];
amino_acid=[];
mutation_type=[];
for i in range(len(protein_seq)):
    codon_pos = i
    wt_aa = protein_seq[codon_pos]

    # add mutations
    for j in AA:
        if wt_aa != j:
            sequential_site.append(codon_pos+19)
            amino_acid.append(j)
            mutation = str(wt_aa) + str(codon_pos+19) + str(j)

            if mutation in list(missing_mutations['mutation']):
                mutation_type.append('spike_in_mutation')
            elif codon_pos in list(missing_sites['AA Position']-19):
                    mutation_type.append('spike_in_site')
            else:
                mutation_type.append('twist_mutation')
   
    # add stops
    if codon_pos in stop_positions:
        sequential_site.append(codon_pos+19);
        amino_acid.append('*');
        mutation_type.append('stop');

missing_mutations.head()

mutation_design_classification = pd.DataFrame(columns=['sequential_site','amino_acid','mutation_type'])
mutation_design_classification['sequential_site']=sequential_site
mutation_design_classification['amino_acid']=amino_acid
mutation_design_classification['mutation_type']=mutation_type

mutation_design_classification.to_csv("testfile.txt", sep='\t')


assert len(missing_sites) == len(
     mutation_design_classification.query('mutation_type == "spike_in_site"')['sequential_site'].unique()
)
assert len(missing_mutations) == len(
     mutation_design_classification.query('mutation_type == "spike_in_mutation"')
)
assert len(mutation_design_classification) == 9634

mutation_design_classification.to_csv('../data/mutation_design_classification.csv', index=False)